# nb_setup_02 — Set / update pipeline config (TEMPLATE — placeholders only)

Run **after `nb_setup_01_bootstrap`** (which creates the `config` Delta table and seeds placeholder
defaults) to point the pipeline at *your* deployed Azure resources.

## ⚠️ Do NOT commit real endpoints
This committed notebook is a **template** and must contain only `<your-…>` placeholders. Real
endpoints are **not stored in the repo** — they live only in the Fabric `config` Delta table.
The recommended workflow:

1. In Fabric, **copy this notebook** to `_local_set_config` (the `notebooks/_local_*.ipynb` name is
   gitignored, so it never gets committed).
2. Edit the `CONFIG` dict in your local copy with your real endpoints (values from
   `infra/deploy.ps1` output).
3. Attach it to the target lakehouse (`aws_connect_lh`) and **Run all**. It performs an **upsert**
   (update existing keys, insert new ones) so it is safe to re-run and only changes the keys you
   list. Because the `config` table MERGE preserves your values across `nb_setup_01` re-runs, you only
   need to set the real endpoints **once**.

Auth is keyless (Entra ID) — no keys are stored; only endpoints + names go in `config`.

> Optional automation: `scripts/set_config.ps1` can read a gitignored `config/config.local.json`
> and write these keys into the Fabric `config` table via a parameterized run — fully repeatable,
> never committed. See `PRODUCT_SPEC.md` § Config hygiene.

In [ ]:
# ============================ EDIT ME (in your gitignored _local_ copy) ============================
# Only the keys you list here are changed; everything else in `config` is left as-is.
# Leave these as <your-...> placeholders in the committed template — fill real values only in the
# gitignored local copy (notebooks/_local_set_config.ipynb).
CONFIG = {
    'doc_intelligence_endpoint': 'https://<your-doc-intelligence>.cognitiveservices.azure.com/',
    'aoai_endpoint':             'https://<your-aoai>.openai.azure.com/',
    'aoai_embedding_deployment': 'text-embedding-3-large',
    'search_endpoint':           'https://<your-search>.search.windows.net',
    'search_index_name':         'docs-rag',
    'kv_name':                   '<your-keyvault-name>',
    'search_key_secret':         'search-admin-key',
    # Optional endpoint pools for scale-out (comma-separated). Leave commented to use the singular
    # endpoints above. See PRODUCT_SPEC.md § Sprint 9 — endpoint pools.
    # 'doc_intelligence_endpoints': 'https://<di-1>.cognitiveservices.azure.com/,https://<di-2>.cognitiveservices.azure.com/',
    # 'aoai_endpoints':            'https://<aoai-1>.openai.azure.com/,https://<aoai-2>.openai.azure.com/',
}
# ==================================================================================================
assert not any('<your-' in str(v) for v in CONFIG.values()), \
    'Refusing to write placeholder values. Fill CONFIG with real endpoints in your _local_ copy first.'

In [ ]:
from datetime import datetime, timezone
from pyspark.sql import Row
from delta.tables import DeltaTable

now = datetime.now(timezone.utc)
rows = [Row(key=k, value=str(v), value_type='string', updated_utc=now) for k, v in CONFIG.items()]
df = spark.createDataFrame(rows)

# Upsert: update keys that exist, insert keys that don't. Other config rows are untouched.
tgt = DeltaTable.forName(spark, 'config')
(tgt.alias('t')
   .merge(df.alias('s'), 't.key = s.key')
   .whenMatchedUpdateAll()
   .whenNotMatchedInsertAll()
   .execute())

print('Applied', len(CONFIG), 'config keys. Current config:')
spark.table('config').orderBy('key').show(50, truncate=False)